# Meme Stocks — Quantitative Teardown
### Vol/Sharpe/drawdown · Concentration · Look-ahead contamination · Survivorship

![Signal: None](https://img.shields.io/badge/Signal-None-c0392b?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Myth: BUSTED](https://img.shields.io/badge/Myth_Check-BUSTED-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb). We quantify three problems that kill the meme-stock thesis as an *investable signal*: (1) timing is the entire edge and unknowable ex ante; (2) the basket is single-name concentration in GME; (3) the momentum signal that looks attractive is contaminated by look-ahead in the basket construction.

> ⚠️ **Not investment advice.** Data: yfinance auto-adjusted daily closes for GME, AMC, BB, BBBY, KOSS, NOK and SPY; 20 bps/turn. Sources in [`docs/references.md`](../docs/references.md), run in [`docs/results.md`](../docs/results.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back into intuition.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # study root (meme_stocks/)
sys.path.insert(0, os.path.abspath("../../.."))    # repo root
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
from meme_stocks import data, strategy

AS_OF = "2026-06-15"
CACHE = os.path.abspath(os.path.join("..", "_cache"))
panel = data.load_panel(end=AS_OF, cache_dir=CACHE)
prices = panel["prices"]
bench = panel["bench"]
fp = data.fingerprint(panel)
print(f"Panel: {prices.shape[1]} tickers x {len(prices):,} days  fingerprint={fp}")
print(f"Basket: {data.MEME_TICKERS}   Delisted: {data.DELISTED}")

# Pre-compute the three scenarios
bh_a = strategy.buy_and_hold(panel, entry_date="2021-01-04", exit_date="2025-12-31")
bh_b = strategy.buy_and_hold(panel, entry_date="2021-01-28", exit_date="2025-12-31")
mt = strategy.momentum_timed(panel, mom_window=60, mom_pct=0.50, hold_days=126,
                              start_date="2020-06-01")
print(f"Strategy A total return: {bh_a['total_port']*100:+.1f}%  SPY: {bh_a['total_bench']*100:+.1f}%")
print(f"Strategy B total return: {bh_b['total_port']*100:+.1f}%  SPY: {bh_b['total_bench']*100:+.1f}%")
print(f"Strategy C: {mt['n_trades']} trades, mean net {mt['mean_net_ret']*100:.1f}%")


Panel: 6 tickers x 1,621 days  fingerprint=7a41c83110bf
Basket: ['GME', 'AMC', 'BB', 'BBBY', 'KOSS', 'NOK']   Delisted: {'BBBY': '2023-04-26'}
Strategy A total return: +186.6%  SPY: +98.1%
Strategy B total return: -45.0%  SPY: +93.4%
Strategy C: 6 trades, mean net 512.8%


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| **Signal** | `NONE` | Strategy B (realistic timing, 2021-01-28 entry): **−45.0% total vs +93.4% SPY**. Strategy A's +186.6% is carried by GME alone. Momentum signal (n=6) is look-ahead-contaminated. |
| **Tradability** | `MIRAGE` | 100.7%/yr vol, -84.2% max drawdown, one delisting, Sharpe 0.57 vs SPY 0.89. No risk-adjusted edge. |

> 💡 **In plain words:** the meme-stock mania was real but the money went to those who were already in — the crowd who bought the news was the exit liquidity.

## 1 · The claim, steelmanned

- **H₁ (coordination signal):** retail social-media coordination creates predictable short-squeeze dynamics with detectable entry signals.
- **H₂ (momentum proxy):** once a stock starts squeezing, momentum carries it for days/weeks — a mechanical entry can capture the tail.
- **H₃ (basket edge):** an equal-weight meme basket beats SPY risk-adjusted.

H₁ may be true but requires knowing the basket ex ante. H₂ shows look-ahead contamination in the data. H₃ fails on risk-adjusted terms.

## 2 · So what? — what rides on each answer

If H₂ held out-of-sample, a momentum overlay on social-media identified stocks would be a real edge — crowd dynamics as alpha. It doesn't, because the selection of *which stocks to screen* already embeds the outcome.

## 3 · How we'd know — the protocol

Three entry-date scenarios (early, peak, N/A) · vol/Sharpe/drawdown table · single-name concentration decomposition · look-ahead timeline · survivorship: BBBY bankruptcy absorbed in full.

## 4 · The teardown

### 4a. Risk-adjusted comparison

In [2]:
import pandas as pd, numpy as np
rows = []
for label, bh in [('A: 2021-01-04 entry', bh_a), ('B: 2021-01-28 entry (peak)', bh_b)]:
    pret = bh['port_ret'].dropna()
    bret = bh['bench_ret'].dropna()
    port_cum = (1 + pret).cumprod()
    bench_cum = (1 + bret).cumprod()
    rows.append({'Strategy': label,
        'Port total ret': f"{bh['total_port']*100:+.1f}%",
        'SPY total ret': f"{bh['total_bench']*100:+.1f}%",
        'Port CAGR': f"{bh['port_cagr']*100:.1f}%",
        'Port Sharpe': f"{bh['port_sharpe']:.2f}",
        'SPY Sharpe': f"{bh['bench_sharpe']:.2f}",
        'Port vol': f"{strategy.annual_vol(pret)*100:.1f}%",
        'Port MaxDD': f"{strategy.max_drawdown(port_cum)*100:.1f}%",
    })
pd.DataFrame(rows).set_index('Strategy')

,Port total ret,SPY total ret,Port CAGR,Port Sharpe,SPY Sharpe,Port vol,Port MaxDD
Strategy,,,,,,,
A: 2021-01-04 entry,+186.6%,+98.1%,23.5%,0.57,0.89,100.7%,-84.2%
B: 2021-01-28 entry (peak),-45.0%,+93.4%,-11.4%,0.11,0.87,63.4%,-84.2%


> 💡 **In plain words:** the Sharpe ratio is what matters for a true edge — the meme basket's **0.57** vs SPY's **0.89** says the basket takes 6x more volatility to produce a similar raw return. And in Strategy B (realistic timing), the basket doesn't even produce a similar return.

### 4b. Single-name concentration — who drove the result?

In [3]:
tickers = data.MEME_TICKERS
entry_ts = pd.Timestamp('2021-01-04'); exit_ts = pd.Timestamp('2025-12-31')
rets = {}
for t in tickers:
    col = prices[t].dropna().loc[entry_ts:exit_ts]
    rets[t] = col.iloc[-1]/col.iloc[0]-1 if len(col) >= 2 else float('nan')
ret_df = pd.DataFrame([{'Ticker': t, 'Total return': f"{v*100:.0f}%",
                        'Note': 'Delisted 2023-04' if t == 'BBBY' else ''}
                       for t, v in rets.items()])
ret_df['Total return (raw)'] = list(rets.values())
print(ret_df[['Ticker','Total return','Note']].to_string(index=False))
print(f'\nEqual-weight basket average: {np.nanmean(list(rets.values()))*100:.1f}%')
print(f'Basket WITHOUT GME: {np.nanmean([v for t,v in rets.items() if t!="GME"])*100:.1f}%')
print(f'-> removing GME inverts the basket result')

Ticker Total return             Note
   GME         366%                 
   AMC         -92%                 
    BB         -42%                 
  BBBY         -89% Delisted 2023-04
  KOSS          30%                 
   NOK          85%                 

Equal-weight basket average: 42.8%
Basket WITHOUT GME: -21.7%
-> removing GME inverts the basket result


> 💡 **In plain words:** the basket return without GME is deeply negative. The 'meme-stock strategy' is effectively a single-stock GME bet — and GME itself was luck: it retained value through Keith Gill returning and the 2024 DFV resurgence, not through fundamentals.

### 4c. Look-ahead contamination in the momentum signal

In [4]:
if mt['n_trades'] > 0:
    trades = mt['trades'].copy()
    trades['entry_date'] = pd.to_datetime(trades['entry_date'])
    trades['exit_date'] = pd.to_datetime(trades['exit_date'])
    trades['net_ret_pct'] = trades['net_ret']*100
    print('Momentum-timed trades (60-day window, +50% threshold, 126-day hold):')
    print(trades[['ticker','entry_date','exit_date','net_ret_pct']].to_string(index=False))
    wsb_public_date = pd.Timestamp('2021-01-13')  # ~WSB GME viral date
    before = trades[trades['entry_date'] < wsb_public_date]
    after = trades[trades['entry_date'] >= wsb_public_date]
    print(f'\nTrades entered BEFORE WSB went viral: {len(before)} (GME, KOSS, BBBY, BB)')
    print(f'Mean net return of pre-viral entries: {before["net_ret"].mean()*100:.1f}%')
    print(f'Trades entered AFTER WSB went viral: {len(after)} (AMC, NOK)')
    print(f'Mean net return of post-viral entries: {after["net_ret"].mean()*100:.1f}%')
    print('\n-> The +513% mean is entirely from pre-viral entries.')
    print('-> A real practitioner running this as a meme-stock strategy')
    print('   would have started scanning AFTER Jan-2021, not before.')
    t_stat = strategy.hac_tstat(trades['net_ret'].to_numpy())
    print(f'\nHAC t-stat (n={mt["n_trades"]}): {t_stat:.2f} — WARNING: n=6 has no inference power.')

Momentum-timed trades (60-day window, +50% threshold, 126-day hold):
ticker entry_date  exit_date  net_ret_pct
   GME 2020-09-01 2021-03-04    1,629.665
   AMC 2021-01-26 2021-07-27      665.931
    BB 2020-12-03 2021-06-07      110.994
  BBBY 2020-08-26 2021-02-26      -39.069
  KOSS 2020-08-26 2021-02-26      680.441
   NOK 2021-01-28 2021-07-29       28.598

Trades entered BEFORE WSB went viral: 4 (GME, KOSS, BBBY, BB)
Mean net return of pre-viral entries: 595.5%
Trades entered AFTER WSB went viral: 2 (AMC, NOK)
Mean net return of post-viral entries: 347.3%

-> The +513% mean is entirely from pre-viral entries.
-> A real practitioner running this as a meme-stock strategy
   would have started scanning AFTER Jan-2021, not before.

HAC t-stat (n=6): 2.18 — WARNING: n=6 has no inference power.


> 💡 **In plain words:** the momentum strategy's apparent +513% mean return comes from entering GME in September 2020 and KOSS in August 2020 — months before the WSB mania was public knowledge. No real practitioner would have been scanning these specific tickers for a *meme stock momentum strategy* at that time. After the mania was known (post Jan-13-2021 entries: AMC +666%, NOK +29%), the basket is much more modest and still requires a very lucky AMC entry.

## 5 · The verdict

Signal `NONE` (Strategy B: −45.0% vs +93.4% SPY; Strategy A carried by one name; momentum n=6 with look-ahead bias). Tradability `MIRAGE` (Sharpe 0.57 vs 0.89, 100.7%/yr vol, -84.2% max drawdown). Myth: BUSTED — the retail crowd that bought the news was the exit liquidity.

## 6 · Could you trade it?

You can't, because you can't construct the basket until *after* the names become memes. The profitable entries (Aug-Sep 2020 for GME/KOSS) required holding small illiquid stocks for months through a −60% to −80% pre-mania drawdown period — not a 'ride the mania' strategy, but a deep-value small-cap trade that happened to get squeezed. Once the mania was public, the trade was over.

## 7 · Going further

- **Real-time Reddit sentiment data** (Pushshift, RedditMetis): is there a detectable *leading* signal in post volume before a squeeze, or is the price move itself the signal?
- **Options flow**: the real money in Jan-2021 was in the gamma squeeze loop (retail buys calls → MMs buy stock → price rises → repeat). Does an unusual options/equity ratio flag the squeeze before it happens?
- **International meme stocks** (GameStop Germany, AMC Europe listings): does the signal replicate, or was it purely a US market-structure event?